# PolypDB Reproduction — Phase 2 Training
U-Net, DeepLabV3+, YOLOv8 on WLI. Expected runtime ~3 hrs on T4.

In [ ]:
# Clone repo and install packages
!git clone https://github.com/christopherh-88/mlrc-PolypDB.git /kaggle/working/mlrc-PolypDB
%cd /kaggle/working/mlrc-PolypDB
!pip install -q segmentation-models-pytorch albumentations pycocotools torchmetrics ultralytics

In [ ]:
import os, sys
os.chdir('/kaggle/working/mlrc-PolypDB')
sys.path.insert(0, '/kaggle/working/mlrc-PolypDB')

# WLI.zip uploads as /kaggle/input/polypdb-wli/WLI/
os.environ['POLYPDB_DATA_ROOT'] = '/kaggle/input/polypdb-wli'
os.environ['POLYPDB_COCO_ROOT'] = '/kaggle/working/mlrc-PolypDB/coco_labels'

# Verify data is accessible
from datasets.polyp_dataset import PolypDataset
for split in ['train', 'val', 'test']:
    ds = PolypDataset('WLI', split)
    print(f'WLI {split}: {len(ds)} samples')

In [ ]:
# Generate YOLO labels
!python scripts/coco_to_yolo.py --modality WLI

# Fix YAML to use absolute Kaggle path
from pathlib import Path
yaml = Path('configs/wli_detection.yaml')
yaml.write_text(f"""path: /kaggle/working/mlrc-PolypDB/yolo_labels/WLI
train: train/images
val:   val/images
test:  test/images
nc: 1
names: ['polyp']
""")
print('YOLO labels ready')

In [ ]:
# Train U-Net (~60 min on T4)
!python train_segmentation.py --model unet --modality WLI --epochs 100 --batch_size 16 --num_workers 2

In [ ]:
# Train DeepLabV3+ (~75 min on T4)
!python train_segmentation.py --model deeplabv3plus --modality WLI --epochs 100 --batch_size 16 --num_workers 2

In [ ]:
# Train YOLOv8 (~40 min on T4)
!python train_detection.py --modality WLI --epochs 100 --batch_size 16

In [ ]:
# Print results table
!python print_results.py

In [ ]:
# Collect all weights into one place for easy download
import shutil
from pathlib import Path

out = Path('/kaggle/working/polypdb_weights')
out.mkdir(exist_ok=True)

for f in Path('results').rglob('*.pth'):
    shutil.copy(f, out / f'{f.parent.name}_{f.name}')
for f in Path('results').rglob('results.json'):
    shutil.copy(f, out / f'{f.parent.name}_results.json')

yolo_best = Path('runs/detect/yolov8s_WLI/weights/best.pt')
if yolo_best.exists():
    shutil.copy(yolo_best, out / 'yolov8s_WLI_best.pt')

print('Files ready to download:')
for f in sorted(out.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')